# TTT paper — cloud runs v2 (revision experiments)
One click: **Runtime → Run all**. Everything is embedded — no uploads needed.

Runs, in order:
1. **E1** five-arm ablation ladder on TTT-Linear-760M: descent / Hebbian / **Hebbian norm-matched** / ascent / frozen (~1.5–2.5 h on a T4; the headline run).
2. **E3** layer-bootstrap confidence intervals for Spearman ρ(CV(γ²), live corrective fraction), from the ladder's own JSON (seconds).
3. **E6 (optional, default ON)** live delta-fraction probe on TTT-Linear-1.3B (~20 min) and TTT-MLP-760M (~20 min), plus the five-arm ladder on 1.3B (~3 h; set `RUN_13B_LADDER = False` to skip).

Results auto-download as `ttcl_cloud_results_v2.zip` at the end (also in the Files pane).
Use a GPU runtime (Runtime → Change runtime type → T4 or better). Keep the tab open on free Colab;
if the runtime dies, re-run — every step caches to disk and skips what is already done.


In [ ]:
# 1. Unpack the embedded validation kit v2 (62 KB, base64)
import base64, os
KIT_B64 = """
...
"""
open("validation_kit.tar.gz","wb").write(base64.b64decode("".join(KIT_B64.split())))
os.system("tar xzf validation_kit.tar.gz")
print(sorted(os.listdir("validation_kit")))
print(open("validation_kit/MANIFEST.txt").read()[-400:])


In [ ]:
# 2. Dependencies (torch is preinstalled on Colab)
%pip -q install "transformers==4.44.2" safetensors msgpack huggingface_hub
import torch, os
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")
RUN_13B_PROBE  = True    # E6: live probe on TTT-Linear-1.3B (~20 min on T4)
RUN_MLP_PROBE  = True    # E6: live probe on TTT-MLP-760M   (~20 min on T4)
RUN_13B_LADDER = True    # E6: five-arm ladder on 1.3B (~3 h on T4); set False to skip
os.makedirs("results", exist_ok=True)


In [ ]:
# 3. Download checkpoints (public). 760M ~3 GB; 1.3B ~5 GB; MLP-760M ~3 GB.
from huggingface_hub import snapshot_download
CKPTS = {"linear-760m": "Test-Time-Training/ttt-linear-760m-pile-8k"}
if RUN_13B_PROBE or RUN_13B_LADDER:
    CKPTS["linear-1.3b"] = "Test-Time-Training/ttt-linear-1.3b-pile-8k"
if RUN_MLP_PROBE:
    CKPTS["mlp-760m"] = "Test-Time-Training/ttt-mlp-760m-pile-8k"
paths = {}
for name, repo in CKPTS.items():
    paths[name] = snapshot_download(repo_id=repo, local_dir=f"ckpts/{name}")
    print("ok", name)


In [ ]:
# 4. Corpus (public-domain novel, same as the workshop run) + un-gated Llama-2 tokenizer mirror
import urllib.request, os
if not os.path.exists("corpus.txt"):
    urllib.request.urlretrieve(
      "https://raw.githubusercontent.com/mmcky/nyu-econ-370/master/notebooks/data/book-war-and-peace.txt",
      "corpus.txt")
TOK = "NousResearch/Llama-2-7b-hf"
print("corpus bytes:", os.path.getsize("corpus.txt"))


In [ ]:
# 5. Convert checkpoints to the PyTorch reference format (cached)
import subprocess, sys, os
def run(cmd, cwd="validation_kit"):
    print("+", " ".join(cmd), flush=True); r = subprocess.run(cmd, cwd=cwd)
    assert r.returncode == 0, cmd
for name in CKPTS:
    if os.path.exists(f"converted/{name}/config.json"):
        print("cached", name); continue
    run([sys.executable, "convert_jax_to_pt.py", f"../ckpts/{name}/streaming_train_state",
         f"../converted/{name}", "--verify"])
print("all converted")


## E1 — five-arm ladder on TTT-Linear-760M
Same harness as the workshop run plus the `hebbian_nm` arm: the error-correction term is deleted
and the remaining additive gradient is rescaled **per token** so every rank-one fast-weight write
keeps the Frobenius norm of the unmodified write. The applied ratio ‖g_desc‖/‖g_hebb‖ is logged
(≈1.04 if the two components are orthogonal at a 27% corrective share, ≈0.79 if opposed, ≈1.37 if parallel).

In [ ]:
# 6. E1 — five-arm ablation ladder on TTT-Linear-760M (headline run)
import subprocess, sys, os, json
out = "results/ladder_linear760m/ablation_results_linear760m.json"
if os.path.exists(out):
    print("cached:", out)
else:
    run([sys.executable, "run_ablation_ladder.py",
        "--checkpoint", "../converted/linear-760m", "--model", "ttt_linear",
        "--tokenizer", TOK, "--text", "../corpus.txt",
        "--max-length", "2048", "--max-tokens", "100000",
        "--arms", "descent,hebbian,hebbian_nm,ascent,frozen",
        "--hebbian-mode", "drop_pred", "--normmatch-level", "token",
        "--device", DEVICE, "--tag", "linear760m", "--out-dir", "../results/ladder_linear760m"])
d = json.load(open(out))
print({a: round(v["ppl"], 3) for a, v in d["per_arm"].items()})
print("norm-match ratio:", {k: v for k, v in d.get("normmatch_ratio", {}).items() if k != "per_layer"})


In [ ]:
# 7. E3 — bootstrap CI for rho(CV(gamma^2), live corrective fraction), 760M
run([sys.executable, "bootstrap_spearman.py", "../" + out, "--out", "../results/spearman_linear760m.json"])


## E6 — replication on a second checkpoint (optional)
Live probe on TTT-Linear-1.3B and TTT-MLP-760M (512 mini-batches each, same as the paper's 760M probe),
then the five-arm ladder on 1.3B.

In [ ]:
# 8. E6 — live delta-fraction probes (1.3B, MLP-760M)
for name, mtype, flag in [("linear-1.3b", "ttt_linear", RUN_13B_PROBE), ("mlp-760m", "ttt_mlp", RUN_MLP_PROBE)]:
    if not flag: continue
    o = f"results/probe_{name}.json"
    if os.path.exists(o):
        print("cached", o); continue
    run([sys.executable, "run_delta_probe.py",
        "--checkpoint", f"../converted/{name}", "--text", "../corpus.txt",
        "--max-length", "2048", "--windows", "4", "--device", DEVICE,
        "--tokenizer", TOK, "--eta-probe", "--out", "../" + o])
    run([sys.executable, "bootstrap_spearman.py", "../" + o, "--out", f"../results/spearman_{name}.json"])


In [ ]:
# 9. E6 — five-arm ladder on TTT-Linear-1.3B (~3 h on T4)
if RUN_13B_LADDER:
    o = "results/ladder_linear13b/ablation_results_linear13b.json"
    if os.path.exists(o):
        print("cached", o)
    else:
        run([sys.executable, "run_ablation_ladder.py",
            "--checkpoint", "../converted/linear-1.3b", "--model", "ttt_linear",
            "--tokenizer", TOK, "--text", "../corpus.txt",
            "--max-length", "2048", "--max-tokens", "100000",
            "--arms", "descent,hebbian,hebbian_nm,ascent,frozen",
            "--hebbian-mode", "drop_pred", "--normmatch-level", "token",
            "--device", DEVICE, "--tag", "linear13b", "--out-dir", "../results/ladder_linear13b"])
    d = json.load(open(o)); print({a: round(v["ppl"], 3) for a, v in d["per_arm"].items()})


In [ ]:
# 10. Package + download results
import shutil, glob, json
print(glob.glob("results/**", recursive=True))
shutil.make_archive("ttcl_cloud_results_v2", "zip", "results")
try:
    from google.colab import files
    files.download("ttcl_cloud_results_v2.zip")
except Exception as e:
    print("manual download from the Files pane:", e)
